In [ ]:
# import pandas as pd
# name='352'
# long_name='3524m031'
# band='w1'
# table = pd.read_csv('./mached_catalog/'+name+'/'+long_name+'/'+long_name+'_'+ band +'_mached.csv')

In [1]:
import numpy as np
import pandas as pd

def calculate_magnitude(flux):
    if flux <= 0:
        return np.nan
    return 22.5 - 2.5 * np.log10(flux)

def calculate_error(flux,dflux):
    mag_upper = calculate_magnitude(flux - dflux)
    mag_lower = calculate_magnitude(flux + dflux)
    dmag = (mag_upper - mag_lower) / 2
    return dmag

cal_mag_ufunc  = np.frompyfunc(calculate_magnitude,1,1)
cal_error_ufunc  = np.frompyfunc(calculate_error,2,1)
def make_single_light_curve(table, index):
    line = table.iloc[index]
    line_len = len(line)
    ra = line[0]
    dec = line[1]
    flux_unfiltered = line[3:line_len:3]
    flux = np.array(flux_unfiltered[flux_unfiltered.notnull()])
    dflux_unfiltered = line[4:line_len:3]
    dflux = np.array(dflux_unfiltered[dflux_unfiltered.notnull()])
    mjdmean_unfiltered = line[5:line_len:3]
    mjdmean = np.array(mjdmean_unfiltered[mjdmean_unfiltered.notnull()])
    assert len(flux)==len(dflux) and len(dflux)==len(mjdmean), 'light curve uncomplete!'
    mag = cal_mag_ufunc(flux)
    error = cal_error_ufunc(flux,dflux)
    return ra, dec, mag, error, mjdmean,flux, dflux

In [2]:
def reduced_chi_Square(flux,error):
    sigma2 = np.power(error,2)
    mean_flux = np.sum(np.divide(flux,sigma2))/np.sum(np.divide(1,sigma2))
    N = len(flux)
    return np.sum(np.divide(np.power(flux-mean_flux,2),sigma2))/N -1 

def weighted_sigma(mag,error):
    sigma2 = np.power(error,2)
    weights = np.divide(1,sigma2)
    mean_mag = np.sum(np.divide(mag,sigma2))/np.sum(np.divide(1,sigma2))
    return np.sqrt(np.sum(weights)*np.sum(np.multiply(weights,np.power(mag-mean_mag,2)))/(np.sum(weights)**2-np.sum(np.power(weights,2))))

from scipy import stats
def Median_absolute_deviation(mag):
    return stats.median_abs_deviation(mag)

def Interquartile_range(mag):
    return stats.iqr(mag)

def Robust_median_statistic(mag,error):
    N = len(mag)
    median_mag = np.median(mag)
    return np.sum(np.divide(np.abs(mag-median_mag),error))/(N-1)

def Normalized_excess_variance(mag,error):
    """
    Calculate the normalized excess variance of a light curve.
    """
    sigma2 = np.power(error,2)
    mean_mag = np.sum(np.divide(mag,sigma2))/np.sum(np.divide(1,sigma2))
    return np.sum(np.power(mag-mean_mag,2)-sigma2)/(len(mag)*mean_mag**2) 

def Peak_to_peak_variability(mag,error):
    return (np.max(mag-error)-np.min(mag+error))/(np.max(mag-error)+np.min(mag+error))

def Lag_1_autocorrelation(mag,error):
    sigma2 = np.power(error,2)
    mean_mag = np.sum(np.divide(mag,sigma2))/np.sum(np.divide(1,sigma2))
    mag_shifted = mag - mean_mag    
    lag_1_autocorr = np.sum(np.multiply(mag_shifted[1:],mag_shifted[:-1]))/np.sum(mag_shifted**2)  
    return lag_1_autocorr    

def Stetson_J(mag,error):
    """
    Stetson J index
    """    
    sigma2 = np.power(error,2)
    mean_mag = np.sum(np.divide(mag,sigma2))/np.sum(np.divide(1,sigma2))
    N = len(mag)
    delta = np.divide(np.sqrt(N/(N-1))*(mag-mean_mag),error)
    product = np.multiply(delta[1:],delta[:-1])
    return np.sum(np.multiply(np.sign(product),np.sqrt(np.abs(product))))

def Stetson_K(mag,error):
    sigma2 = np.power(error,2)
    mean_mag = np.sum(np.divide(mag,sigma2))/np.sum(np.divide(1,sigma2))
    N = len(mag)
    delta = np.divide(np.sqrt(N/(N-1))*(mag-mean_mag),error)
    denominator = np.sqrt(np.sum(np.power(delta,2)*(1/N)))
    return (1/N)*np.sum(np.abs(delta))/denominator

def yita(mag,error):
    sigma2 = np.power(error,2)
    mean_mag = np.sum(np.divide(mag,sigma2))/np.sum(np.divide(1,sigma2))
    return np.sum(np.power(mag[1:]-mag[:-1],2))/np.sum(np.power(mag-mean_mag,2))

In [3]:
import time
import pickle
def one_footprint_cal(name,long_name,band):
    start = time.time()
    table = pd.read_csv('./mached_catalog/'+name+'/'+long_name+'/'+long_name+'_'+ band +'_mached.csv')
    result_list = []
    for i in range(0,len(table)):
        ra, dec, mags, errors, mjdmean, flux, dflux = make_single_light_curve(table,i)
        if len(mags)>=10:
            chii_squre = 0
            mean_mag = np.mean(mags)
            for mag,error in zip(mags,errors):
                chi_squre += ((mag-mean_mag)/error)**2
            reduced_chi_Square = reduced_chi_Square(flux,dflux)
            w_sigma = weighted_sigma(mag,error)
            MAD = Median_absolute_deviation(mag)
            IQR = Interquartile_range(mag)
            RmStat = Robust_median_statistic(mag,error)
            Nev = Normalized_excess_variance(mag,error)
            p2pv = Peak_to_peak_variability(mag,error)
            L1_acr = Lag_1_autocorrelation(mag,error)
            Stetson_J = Stetson_J(mag,error)
            Stetson_K = Stetson_K(mag,error)           
            yita = yita(mag,error)
            
            result_line = np.array([i,ra,dec,mean_mag,chi_squre,reduced_chi_Square,w_sigma,MAD,IQR,RmStat,Nev,p2pv,L1_acr,Stetson_J,Stetson_K,yita])
            result_list.append(result_line)

        # result_list.append([ra, dec, mags, errors, mjdmean])
    # fw = open('./mached_catalog/'+name+'/'+long_name+'/'+long_name+'_'+ band +'_light_curves.dat','wb')
    # pickle.dump(result_list,fw)

    result_table = pd.DataFrame(np.array(result_list),columns=['id_in_matched','ra','dec','Mean','kai_squre','reduced_chi_Square','w_sigma','MAD','IQR','RmStat','Nev','p2pv','L1_acr','Stetson_J','Stetson_K','yita'])
    print(result_table)
    result_table.to_csv('./mached_catalog/'+name+'/'+long_name+'/'+long_name+'_'+ band +'new_features.csv')
    end = time.time()
    print('Task %s runs %0.2f seconds.' % (long_name, (end - start)))

In [ ]:
one_footprint_cal('000','0000m016','w1')

In [4]:
from multiprocessing import Pool
import time
import glob
import os

thread_pool = Pool()
first_layer_names = glob.glob('[0-9][0-9][0-9]',root_dir='./mached_catalog')
for name in first_layer_names:
        second_layer_names = glob.glob('[0-9][0-9][0-9][0-9]*',
                                    root_dir='./mached_catalog/'+name)
        for long_name in second_layer_names:
            for band in ('w1', 'w2'):
                if True == os.path.isfile('./mached_catalog/'+name+'/'+long_name+'/'+long_name+'_'+ band +'_mached.csv'):
                    thread_pool.apply_async(one_footprint_cal,args=(name,long_name,band))
print('Waiting for all subprocesses done...')
thread_pool.close()
thread_pool.join()
print('all subprocesses done')

Waiting for all subprocesses done...


/tmp/ipykernel_115617/1321680839.py:5: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  table = pd.read_csv('./mached_catalog/'+name+'/'+long_name+'/'+long_name+'_'+ band +'_mached.csv')
/tmp/ipykernel_115617/1321680839.py:5: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  table = pd.read_csv('./mached_catalog/'+name+'/'+long_name+'/'+long_name+'_'+ band +'_mached.csv')
/tmp/ipykernel_115617/1321680839.py:5: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  table = pd.read_csv('./mached_catalog/'+name+'/'+long_name+'/'+long_name+'_'+ band +'_mached.csv')
/tmp/ipykernel_115617/1321680839.py:5: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  table = pd.read_csv('./mached_catalog/'+name+'/'+long_name+'/'+long_name+'_'+ band +'_mached.csv')


In [ ]:
# import matplotlib.pyplot as plt
# plt.hist(np.array(result_table['kai_squre']),bins=100,range=(0,100))